In [ ]:
#!pip install openai

# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 03 Step 3 — Rank Generated Completions with a Judge Model

## Purpose
This notebook evaluates and ranks the candidate policy completions generated in Step 2.

## Why this step is important
GRPO training requires preference information. This notebook converts raw completions into ranked groups by scoring them with a judge model using the rubric based framework provided by the project.

## Inputs
- Completion groups from `outputs/completions/`
- OpenAI API key from `.env`

## Outputs
- Ranked completion groups saved to `outputs/rankings/`

## Main tasks
1. Load the generated completions
2. Send completions to the judge model
3. Score outputs according to the rubric
4. Rank completions from strongest to weakest
5. Save the ranked results for training

## Success criteria
This step is complete when each prompt has an ordered set of ranked completions saved in the rankings folder.

In [ ]:
from openai import OpenAI
import pandas as pd
import tqdm

In [ ]:
import json
import pandas as pd

def to_frame_from_jsonl_safe(path):
    rows, bad = [], []
    # 'utf-8-sig' strips a UTF-8 BOM if present
    with open(path, 'r', encoding='utf-8-sig', errors='strict') as f:
        for i, line in enumerate(f, 1):
            raw = line.strip()
            if not raw:  # skip empty/whitespace-only lines
                continue
            try:
                # Handle the (rare) case where a line is a JSON array: [ {...}, {...} ]
                if raw.startswith('[') and raw.endswith(']'):
                    arr = json.loads(raw)
                    if isinstance(arr, list):
                        rows.extend(arr)
                    else:
                        rows.append(arr)
                else:
                    rows.append(json.loads(raw))
            except json.JSONDecodeError as e:
                bad.append((i, str(e), raw[:120]))
                # continue to skip malformed line(s)
                continue

    if bad:
        print(f"⚠️ Skipped {len(bad)} malformed line(s). Examples:\n" +
              "\n".join([f"  line {ln}: {err} | preview: {preview!r}" for ln, err, preview in bad[:5]]))
    return pd.DataFrame(rows)

df_jsoll = to_frame_from_jsonl_safe('/content/cyber_policies_4comps.jsonl')
df_jsoll.head()


⚠️ Skipped 1 malformed line(s). Examples:
  line 26: Expecting value: line 1 column 1 (char 0) | preview: '\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


,prompt_id,prompt,rag_topk,rag_chunks,completions,scores,ranked_indices,ranked_scores
0,6f161b6a8eea,Generate a cybersecurity policy for a company ...,20,[{'text': 'hold back on transitioning to the c...,[{'text': '**Cybersecurity Policy for [Company...,None,None,None
1,96ac553b335b,Generate a cybersecurity policy for a small (l...,20,[{'text': 'NIST CSWP 29 The NIST Cybersecurit...,[{'text': '**Cybersecurity Policy for [Company...,None,None,None
2,bf4dd0beeb56,Generate a cybersecurity policy for a company ...,20,[{'text': '| A COMPREHENSIVE GUIDE TO OT SECU...,[{'text': '**Cybersecurity Policy for Agricult...,None,None,None
3,585c9c0287bb,Generate a cybersecurity policy for a large co...,20,[{'text': 'hold back on transitioning to the c...,[{'text': 'Based on the provided query and the...,None,None,None
4,7d92351eb5ec,Generate a cybersecurity policy for a company ...,20,"[{'text': 'descriptions of the changes, amon...",[{'text': '**Cybersecurity Policy for [Company...,None,None,None


In [ ]:
# Keep only prompt, rag, completions
df_jsoll = df_jsoll[['prompt_id',	'prompt','completions']]

# Check the first few rows
df_jsoll.head()

,prompt_id,prompt,completions
0,6f161b6a8eea,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for [Company...
1,96ac553b335b,Generate a cybersecurity policy for a small (l...,[{'text': '**Cybersecurity Policy for [Company...
2,bf4dd0beeb56,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for Agricult...
3,585c9c0287bb,Generate a cybersecurity policy for a large co...,[{'text': 'Based on the provided query and the...
4,7d92351eb5ec,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for [Company...


In [ ]:
# Initialize the OpenAI client

api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not set in environment variables")

client = OpenAI(api_key=api_key)

In [ ]:
# === GRPO-ready ranking with GPT-4o (single cell/script) ===
# pip install -U openai pandas tqdm

import os, json, time
from typing import Any, List, Dict, Tuple
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ---------------- CONFIG ----------------
OPENAI_MODEL = "gpt-4o"
INPUT_JSONL  = "/content/cyber_policies_4comps.jsonl"   # used only if df_jsoll not preloaded
OUT_PATH     = "/content/cyber_policies_4comps-grpo_ranked.jsonl"
MAX_RETRIES  = 2
SLEEP_S      = 0.4

# ---------------- RUBRIC ----------------
RUBRIC = """
You are an expert cybersecurity policy evaluator.

Evaluate EACH completion on these criteria (integers only, 1–3 each):
1) Comprehensive Scope: technical, physical, administrative coverage
2) Clear & Specific Guidelines: clarity, specificity, actionability
3) User Awareness & Training: frequency, depth, structure
4) Incident Response Plan: completeness, accessibility, testing cadence

Total = sum of the four criteria ∈ [4..12].
Return DISTINCT totals across the completions (no ties). If any tie occurs, break the tie strictly.

Respond ONLY with JSON:
{
  "scores": [
    {"index": 0, "comprehensive_scope": int, "clear_specific": int, "awareness_training": int, "incident_response": int, "total": int, "brief_reason": "…"},
    {"index": 1, ...}
  ],
  "ranking": [best_index, second_index, ...]
}
"""

# ---------------- HELPERS ----------------
def load_jsonl_safe(path: str) -> pd.DataFrame:
    rows, bad = [], []
    with open(path, 'r', encoding='utf-8-sig') as f:
        for i, line in enumerate(f, 1):
            s = line.strip()
            if not s:
                continue
            try:
                rows.append(json.loads(s))
            except json.JSONDecodeError as e:
                bad.append((i, str(e)))
    if bad:
        print(f"⚠️ Skipped {len(bad)} malformed line(s). Examples:", bad[:3])
    return pd.DataFrame(rows)

def to_texts(comp_list: Any) -> List[str]:
    out = []
    if isinstance(comp_list, list):
        for c in comp_list:
            if isinstance(c, dict) and "text" in c:
                out.append(str(c["text"]))
            else:
                out.append(str(c))
    return out

def ask_judge(client, prompt, comps, max_retries=2, sleep_s=0.4):
    payload = {"prompt": prompt, "completions": [{"index": i, "text": comps[i]} for i in range(len(comps))]}
    last_text = None
    for _ in range(max_retries + 1):
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            response_format={"type": "json_object"},
            temperature=0,
            messages=[
                {"role": "system", "content": RUBRIC},
                {"role": "user",   "content": json.dumps(payload, ensure_ascii=False)},
            ],
        )
        text = resp.choices[0].message.content
        try:
            obj = json.loads(text)
            totals  = [s.get("total") for s in obj.get("scores", [])]
            ranking = obj.get("ranking", [])

            # REQUIREMENTS for GRPO:
            # 1) We have as many scores as comps
            # 2) totals are ints in [4..12]
            # 3) We have a full ranking of all indices
            # (No longer require all totals to be distinct)
            ok_totals = (len(totals) == len(comps) and all(isinstance(t, int) and 4 <= t <= 12 for t in totals))
            ok_rank   = (isinstance(ranking, list) and len(ranking) == len(comps) and set(ranking) == set(range(len(comps))))
            if ok_totals and ok_rank:
                return obj

            last_text = text
        except Exception:
            last_text = text
        time.sleep(sleep_s)
    # If judge returns valid ranking but we still failed validation above, surface last JSON
    raise RuntimeError(f"Judge failed validation. Last response: {last_text}")

def aligned_ranks_from_ranking(ranking: List[int], n: int) -> List[int]:
    """ranking [best→worst indices] -> ranks aligned to original order (1..N)."""
    m = {idx: r+1 for r, idx in enumerate(ranking)}
    return [m.get(i, None) for i in range(n)]

def preferences_from_ranking(ranking: List[int]) -> List[Tuple[int, int]]:
    """
    Produce all pairwise preferences (winner, loser) from a total order.
    e.g., [2,0,3,1] -> (2>0), (2>3), (2>1), (0>3), (0>1), (3>1)
    """
    prefs = []
    for i in range(len(ranking)):
        for j in range(i+1, len(ranking)):
            prefs.append((ranking[i], ranking[j]))
    return prefs

# ---------------- LOAD DATA ----------------
if "df_jsoll" in globals() and isinstance(globals()["df_jsoll"], pd.DataFrame):
    df = globals()["df_jsoll"].copy()
else:
    df = load_jsonl_safe(INPUT_JSONL)

# Keep only required columns; create missing if absent
for col in ["prompt_id", "prompt", "completions"]:
    if col not in df.columns:
        df[col] = None
df = df[["prompt_id", "prompt", "completions"]]

# Filter to rows with at least 2 completions (ideally 4 for GRPO)
def has_two_plus(x): return isinstance(x, list) and len(x) >= 2
df = df[df["completions"].apply(has_two_plus)].reset_index(drop=True)

# ---------------- RANK WITH GPT-4o ----------------

out_f = open(OUT_PATH, "w", encoding="utf-8")

skipped = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Judging & writing GRPO JSONL"):
    pid = row["prompt_id"]
    prompt = (row["prompt"] or "") if pd.notna(row["prompt"]) else ""
    comps  = to_texts(row["completions"])

    # Enforce exactly 4 for the pipeline (skip others). Change as needed.
    if len(comps) != 4:
        skipped += 1
        continue

    judged = ask_judge(client, prompt, comps, max_retries=MAX_RETRIES, sleep_s=SLEEP_S)
    ranking = judged["ranking"]                               # best→worst indices
    aligned = aligned_ranks_from_ranking(ranking, len(comps)) # aligned to original order
    prefs   = preferences_from_ranking(ranking)               # all pairwise winner>loser

    # Build GRPO-ready record
    record = {
        "prompt_id": pid,
        "prompt": prompt,
        "completions": row["completions"],   # keep original structure
        "scores": judged["scores"],          # rubric breakdown + total per completion
        "ranking": ranking,                  # best→worst indices
        "aligned_ranks": aligned,            # aligned to original order (1..4)
        "preferences": [{"winner": w, "loser": l} for (w, l) in prefs],
        # Helpful extras:
        "top_total": max(s["total"] for s in judged["scores"]),
        "second_total": sorted([s["total"] for s in judged["scores"]], reverse=True)[1],
    }

    out_f.write(json.dumps(record, ensure_ascii=False) + "\n")

out_f.close()
print(f"✅ Wrote GRPO JSONL: {OUT_PATH}")
if skipped:
    print(f"ℹ️ Skipped {skipped} row(s) that did not have exactly 4 completions.")


Judging & writing GRPO JSONL: 100%|██████████| 44/44 [09:02<00:00, 12.33s/it]

✅ Wrote GRPO JSONL: /content/cyber_policies_4comps-2_grpo_ranked.jsonl


In [ ]:
df_ranked = pd.read_json("/content/cyber_policies_4comps-2_grpo_ranked.jsonl", lines=True)
df_ranked.head()

,prompt_id,prompt,completions,scores,ranking,aligned_ranks,preferences,top_total,second_total
0,6f161b6a8eea,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for [Company...,"[{'index': 0, 'comprehensive_scope': 3, 'clear...","[2, 3, 0, 1]","[3, 4, 1, 2]","[{'winner': 2, 'loser': 3}, {'winner': 2, 'los...",12,11
1,96ac553b335b,Generate a cybersecurity policy for a small (l...,[{'text': '**Cybersecurity Policy for [Company...,"[{'index': 0, 'comprehensive_scope': 2, 'clear...","[2, 3, 1, 0]","[4, 3, 1, 2]","[{'winner': 2, 'loser': 3}, {'winner': 2, 'los...",12,11
2,bf4dd0beeb56,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for Agricult...,"[{'index': 0, 'comprehensive_scope': 3, 'clear...","[2, 0, 1, 3]","[2, 3, 1, 4]","[{'winner': 2, 'loser': 0}, {'winner': 2, 'los...",12,11
3,585c9c0287bb,Generate a cybersecurity policy for a large co...,[{'text': 'Based on the provided query and the...,"[{'index': 0, 'comprehensive_scope': 3, 'clear...","[3, 1, 2, 0]","[4, 2, 3, 1]","[{'winner': 3, 'loser': 1}, {'winner': 3, 'los...",12,11
4,7d92351eb5ec,Generate a cybersecurity policy for a company ...,[{'text': '**Cybersecurity Policy for [Company...,"[{'index': 0, 'comprehensive_scope': 3, 'clear...","[2, 0, 3, 1]","[2, 4, 1, 3]","[{'winner': 2, 'loser': 0}, {'winner': 2, 'los...",12,10
